In [9]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
temas = {
    "Saúde": "Saúde pública, medicamentos, hospitais, SUS, planos de saúde",
    "Tributário": "Impostos, tributos, arrecadação, reforma tributária, ICMS, IR",
    "Trabalho": "Emprego, CLT, reforma trabalhista, salário mínimo, sindicatos",
    "Tecnologia": "Inteligência artificial, dados, internet, telecomunicações, inovação",
    "Meio Ambiente": "Clima, desmatamento, sustentabilidade, agrotóxicos, recursos hídricos",
    "Educação": "Ensino, escolas, universidades, MEC, bolsas de estudo",
    "Segurança": "Polícia, violência, armamento, presídios, crime organizado",
    "Infraestrutura": "Estradas, saneamento, habitação, transporte, obras públicas",
    "Economia": "PIB, inflação, juros, orçamento, dívida pública, privatização",
    "Direitos Civis": "Direitos humanos, igualdade, minorias, liberdade, cidadania"
}

In [11]:
def gerar_embedding(texto):
    resposta = client.embeddings.create(
        input=texto,
        model="text-embedding-3-small"
    )
    return resposta.data[0].embedding

embeddings_temas = {}

for tema, descricao in temas.items():
    embeddings_temas[tema] = gerar_embedding(descricao)
    print(f"Tema embedado: {tema}")

Tema embedado: Saúde
Tema embedado: Tributário
Tema embedado: Trabalho
Tema embedado: Tecnologia
Tema embedado: Meio Ambiente
Tema embedado: Educação
Tema embedado: Segurança
Tema embedado: Infraestrutura
Tema embedado: Economia
Tema embedado: Direitos Civis


In [12]:
import numpy as np

def calcular_similaridade(vetor_a, vetor_b):
    a = np.array(vetor_a)
    b = np.array(vetor_b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def classificar_proposicao(ementa):
    embedding_ementa = gerar_embedding(ementa)
    similaridades = {}
    for tema, vetor_tema in embeddings_temas.items():
        similaridades[tema] = calcular_similaridade(embedding_ementa, vetor_tema)
    return max(similaridades, key=similaridades.get)

In [13]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

df = pd.read_sql("SELECT id, ementa FROM proposicoes LIMIT 10", engine)
df.head()

,id,ementa
0,106701,"Altera a Lei nº 9.605, de 12 de fevereiro de 1..."
1,155019,"Cria o Programa Nacional de Coleta, Armazename..."
2,264726,"Dá nova redação ao art. 15 da Lei nº 10.826, d..."
3,271967,Dispõe sobre a obrigatoriedade da Natureza Púb...
4,318352,Altera o caput e o inciso II do art. 22 da Lei...


In [14]:
for index, linha in df.iterrows():
    tema = classificar_proposicao(linha["ementa"])
    print(f"{linha['id']} → {tema}")

106701 → Tecnologia
155019 → Tributário
264726 → Segurança
271967 → Infraestrutura
318352 → Tributário
327881 → Tributário
328385 → Trabalho
342725 → Educação
377384 → Saúde
399528 → Tributário


In [ ]:
df_total = pd.read_sql("SELECT id, ementa FROM proposicoes", engine)

resultados = []

for index, linha in df_total.iterrows():
    tema = classificar_proposicao(linha["ementa"])
    resultados.append({"id": linha["id"], "tema": tema})
    if index % 100 == 0:
        print(f"{index} de {len(df_total)} processadas")
        with open("data/raw/resultados_classificacao.json", "w", encoding="utf-8") as f:
            json.dump(resultados, f, ensure_ascii=False)

print("Classificação concluída!")

1801
